# BRFSS Diabetes Model Evaluation

دفتر تشغيل Google Colab للمشروع. يحلل السكربت انتشار السكري المُبلغ ذاتيًا ويقارن النماذج؛ لا يشخّص المرض ولا يتنبأ بحدوثه مستقبلًا. يحتاج التحليل إلى ملفات CDC بصيغة SAS XPT لعامي 2024 و2025. لا ترفع بيانات صحية شخصية أو ملفات تحتوي معلومات تعريفية.

**مسار العمل:** تجهيز مستودع المشروع ← تثبيت المتطلبات ← وضع ملفات CDC في مجلد البيانات ← تشغيل الاختبارات والتحليل ← استعراض النتائج.

## 1. تجهيز المشروع

بعد نشر التحديثات على GitHub، شغّل الخلية التالية لاستنساخ المستودع. عند العمل من نسخة محلية موجودة في `/content`، يمكن تعيين `PROJECT_DIR` يدويًا بدل الاستنساخ.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/alrashedinaifrr/brfss-diabetes-model-evaluation.git'
PROJECT_DIR = Path('/content/brfss-diabetes-model-evaluation')
if not (PROJECT_DIR / 'run_v11.py').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
assert (PROJECT_DIR / 'run_v11.py').exists(), 'لم أجد run_v11.py في جذر المشروع'
print('Project:', PROJECT_DIR)

In [ ]:
%cd /content/brfss-diabetes-model-evaluation
%pip install -q -r requirements.txt

## 2. تجهيز ملفات BRFSS

نزّل ملفي SAS Transport من صفحة CDC الرسمية وضعهما في مجلد على Google Drive باسم `brfss_data`، ثم عدّل المسار أدناه. يجب أن تكون الأسماء `LLCP2024.XPT` و`LLCP2025.XPT`. قراءة الملفات تتم على دفعات لتقليل استهلاك الذاكرة.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = Path('/content/drive/MyDrive/brfss_data')
required = [DATA_DIR / 'LLCP2024.XPT', DATA_DIR / 'LLCP2025.XPT']
for path in required:
    print(('✓' if path.is_file() else '✗'), path)
if not all(p.is_file() for p in required):
    raise FileNotFoundError('ضع ملفي XPT في مجلد brfss_data على Drive أو حدّث DATA_DIR.')

## 3. فحوصات الكود ثم تشغيل التحليل

تتضمن النتائج تقييمًا موزونًا وغير موزون، تحليل عتبات القرار كسيناريوهات، تحليل مجموعات BMI، معايرة، ومقارنات حساسية. أوزان المسح وحدها لا تكفي لإثبات صحة فواصل الثقة التصميمية.

In [ ]:
subprocess.run(['python', '-m', 'pytest', '-q', 'test_v11.py'], cwd=PROJECT_DIR, check=True)
OUTPUT_DIR = Path('/content/brfss_results')
subprocess.run([
    'python', str(PROJECT_DIR / 'run_v11.py'),
    '--data-dir', str(DATA_DIR),
    '--output-dir', str(OUTPUT_DIR),
], check=True)

In [ ]:
import pandas as pd
from IPython.display import display

summary_path = OUTPUT_DIR / 'model_comparison_2025.csv'
display(pd.read_csv(summary_path).head(20))
print('Output files:')
for path in sorted(OUTPUT_DIR.rglob('*')):
    if path.is_file(): print(path.relative_to(OUTPUT_DIR))

In [ ]:
from IPython.display import display, Image
for path in sorted((OUTPUT_DIR / 'figures').glob('*.png')):
    print(path.name)
    display(Image(filename=str(path)))

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive('/content/brfss_results', 'zip', OUTPUT_DIR)
print('Results archive:', archive)
files.download(archive)

## قراءة النتائج

- المقاييس المنشورة من نسخة سابقة ليست بديلًا عن إعادة تشغيل هذا الإصدار؛ قد تتغير النتائج مع تغيّر الكود أو المدخلات.
- التقييم على 2025 استُخدم خلال التطوير، لذلك لا يُعد اختبارًا خارجيًا نهائيًا غير ملموس.
- هذا التحليل استكشافي، وليس أداة طبية أو توصية فحص. راجع `TECHNICAL_REPORT.md` و`VALIDATION_STATUS.md` للمنهج والحدود.